# 🚆 Storytelling with Data: ปริมาณผู้โดยสารระบบขนส่งสาธารณะไทย ปี 2568–2569

> **เป้าหมาย:** วิเคราะห์และเล่าเรื่องข้อมูลปริมาณผู้โดยสารระบบขนส่งสาธารณะของไทย

ตั้งแต่การโหลดข้อมูล → EDA → ทำความสะอาด → วิเคราะห์เชิงลึก → ตอบโจทย์ 3 ข้อหลัก

---

🔗 **แหล่งข้อมูล:** THackle by Big Data Institute (BDI)

📎 https://www.thackle.or.th/th/dataset/94

| ขั้นตอน | หัวข้อ |
|--------|--------|
| 1️⃣ | Setup & Load Data |
| 2️⃣ | First Look (Dataset Overview) |
| 3️⃣ | Data Quality & Cleaning |
| 4️⃣ | Data Preparation (Merge & Feature Engineering) |
| 5️⃣ | Advanced EDA & Insights |
| 5.1 | Q1 — Modal Share: คนไทยเดินทางด้วยอะไรมากที่สุด? |
| 5.2 | Q2 — Urban Rail Comparison: แต่ละสายมีพฤติกรรมต่างกันอย่างไร? |
| 5.3 | Q3 — Event Detection: วันหยุดและเทศกาลเห็นได้ในข้อมูลไหม? |
| 6️⃣ | สรุป Key Insights |

---

📦 **ข้อมูล:** ปริมาณการเดินทางของประชาชนด้วยระบบขนส่งสาธารณะในประเทศไทย  
ครอบคลุมช่วงเวลาประมาณ 14 เดือน (ปี 2568–2569) รวบรวมข้อมูลรายวันจากระบบขนส่งหลายประเภท เช่น

*   รถไฟฟ้า BTS, MRT สายต่าง ๆ
*   Airport Rail Link
*   และรถไฟชานเมืองสายสีแดง


🤔 ก่อนเริ่ม: **ทำไมข้อมูลขนส่งสาธารณะถึงสำคัญ?**

ข้อมูลผู้โดยสารรายวันไม่ใช่แค่ตัวเลข — แต่เป็น **พฤติกรรม** ที่บอกเราว่า:
- คนทำงานออกจากบ้านกี่โมง
- เทศกาลไหนที่คนนิยมเดินทาง
- ระบบขนส่งสายใดกำลังเติบโตหรือหดตัว
- โครงสร้างพื้นฐานคุ้มค่าการลงทุนหรือเปล่า

```
Raw Data → Cleaning → Analysis → Insight → Story → Policy
    ↑                                                  ↑
  ข้อมูลดิบ                              การตัดสินใจเชิงนโยบาย
```


## 1️⃣ Setup & Load Data

### 📦 ติดตั้งและ Import Libraries

ก่อนเริ่มต้น เราต้อง import library ที่จำเป็นสำหรับการวิเคราะห์ข้อมูลและสร้างกราฟ โดย `pandas` ใช้สำหรับจัดการตาราง, `numpy` สำหรับคำนวณเชิงตัวเลข, `matplotlib` และ `seaborn` สำหรับสร้าง static chart, `plotly` สำหรับ interactive visualization

นอกจากนี้เรายังติดตั้ง `ydata-profiling` ซึ่งเป็นเครื่องมือสร้างรายงานคุณภาพข้อมูลแบบอัตโนมัติ — มีประโยชน์มากในช่วง EDA เบื้องต้น

`sns.set_style("whitegrid")` — ตั้งค่ารูปแบบกราฟให้พื้นหลังสีขาวและมีเส้นตาราง เพื่อให้อ่านค่าได้ง่ายขึ้น


In [2]:
!pip install ydata-profiling

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
from ydata_profiling import ProfileReport

sns.set_style("whitegrid")
print("✅ Import สำเร็จ! พร้อมวิเคราะห์ข้อมูลแล้ว 🚀")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.4/400.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 4.6 MB/s eta 0:00:00
✅ Import สำเร็จ! พร้อมวิเคราะห์ข้อมูลแล้ว 🚀


### 📂 โหลดข้อมูล

เราโหลดไฟล์ CSV สองปีพร้อมกัน ได้แก่ `passengers68.csv` (ปี 2568/2025) และ `passengers69.csv` (ปี 2569/2026)

**⚠️ สังเกต:** Pandas แสดง `DtypeWarning` เนื่องจากคอลัมน์บางคอลัมน์มี mixed types — นี่คือสัญญาณแรกว่าข้อมูลดิบยังต้องการการทำความสะอาด เราจะจัดการในขั้นตอนถัดไป


In [3]:
# เปลี่ยน file CSV ให้เป็น DataFrame เพื่อจัดการกับข้อมูลใน runtime

df_2025 = pd.read_csv('passengers68.csv')  # ชุดข้อมูลของปี 2025
df_2026 = pd.read_csv('passengers69.csv')  # ชุดข้อมูลของปี 2026


/tmp/ipykernel_955/81675308.py:3: DtypeWarning: Columns (0,1,2,3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_2025 = pd.read_csv('passengers68.csv')  # ชุดข้อมูลของปี 2025


## 2️⃣ First Look — Dataset Overview

### 📊 ภาพรวมข้อมูลเบื้องต้น

ขั้นตอนแรกของ EDA คือการ 'สำรวจ' ข้อมูลก่อน — เราจะดู:

- **head()** — ดูตัวอย่างข้อมูลว่ามีโครงสร้างอย่างไร
- **shape** — ดูขนาด dataset ว่ามีกี่แถว กี่คอลัมน์
- **info()** — ดู data type และจำนวน non-null values ของแต่ละคอลัมน์
- **describe()** — ดูสถิติพื้นฐาน (สำหรับ categorical columns)

การทำ First Look ช่วยให้เราตั้งสมมติฐานได้ว่าข้อมูลมีปัญหาอะไร และควรโฟกัสวิเคราะห์ตรงไหนก่อน


In [4]:
df_2025.head()  # แสดงข้อมูล 5 แถวแรกของปี 2025

,รูปแบบการเดินทาง,วัตถุประสงค์,สาธารณะ/ส่วนบุคคล,หน่วยงาน,ยานพาหนะ/ท่า,วันที่,หน่วย,ปริมาณ
0,ทางถนน,การเดินทางระหว่างจังหวัด,สาธารณะ,บขส.,รถ บขส. และ รถร่วม,01/01/2025,คน,"127,551"
1,ทางถนน,การเดินทางระหว่างจังหวัด,สาธารณะ,ขบ.,รถหมวด 3,01/01/2025,คน,"8,218"
2,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,ทล.,รถยนต์เฉพาะ 4 ล้อ (10 จุดสำรวจ),01/01/2025,คัน,"877,943"
3,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,ทล.,รถยนต์ทุกประเภท (10 จุดสำรวจ),01/01/2025,คัน,"932,642"
4,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,กทพ.,รถยนต์เฉพาะ 4 ล้อ (ทางด่วน),01/01/2025,คัน,"1,364,992"


In [5]:
df_2026.head()  # แสดงข้อมูล 5 แถวแรกของปี 2026

,รูปแบบการเดินทาง,วัตถุประสงค์,สาธารณะ/ส่วนบุคคล,หน่วยงาน,ยานพาหนะ/ท่า,วันที่,หน่วย,ปริมาณ
0,ทางถนน,การเดินทางระหว่างจังหวัด,สาธารณะ,บขส.,รถ บขส. และ รถร่วม,01/01/2026,คน,"112,325"
1,ทางถนน,การเดินทางระหว่างจังหวัด,สาธารณะ,ขบ.,รถหมวด 3,01/01/2026,คน,0
2,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,ทล.,รถยนต์เฉพาะ 4 ล้อ (10 จุดสำรวจ),01/01/2026,คัน,"892,218"
3,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,ทล.,รถยนต์ทุกประเภท (10 จุดสำรวจ),01/01/2026,คัน,"980,649"
4,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,กทพ.,รถยนต์เฉพาะ 4 ล้อ (ทางด่วน),01/01/2026,คัน,"1,231,605"


### 📐 ตรวจสอบขนาด Dataset

จากผลลัพธ์ด้านบน เราเห็นว่า **ข้อมูลปี 2025 มีแถวมากกว่าปี 2026 อย่างมาก** — เหตุผลหลักคือ ข้อมูลปี 2025 ครอบคลุมเต็มปี (365 วัน) ขณะที่ปี 2026 มีข้อมูลเพียงช่วงต้นปี (ประมาณ 70 วัน) เท่านั้น

สิ่งที่น่าสังเกตอีกอย่างคือ `df_2025` แสดงจำนวนแถวสูงผิดปกติ (69,440) เมื่อเทียบกับ non-null count เพียง 15,696 แถว — นี่แสดงว่ามี **แถวว่าง (empty rows) จำนวนมาก** ซึ่งต้องกำจัดออกในขั้นตอน cleaning


In [6]:
print(f"ปี 2025 → {df_2025.shape[0]:,} แถว × {df_2025.shape[1]} คอลัมน์")
print(f"ปี 2026 → {df_2026.shape[0]:,} แถว × {df_2026.shape[1]} คอลัมน์")


ปี 2025 → 69,440 แถว × 8 คอลัมน์
ปี 2026 → 3,010 แถว × 8 คอลัมน์


### 🔬 ตรวจสอบ Data Types และ Missing Values

`info()` บอกเราว่า:
1. **ทุก column เป็น object** — รวมถึงคอลัมน์ `ปริมาณ` ที่ควรเป็นตัวเลข แต่ถูก Pandas อ่านเป็น string เพราะมีเครื่องหมาย comma เช่น `"127,551"` — ต้องแปลงในขั้นตอน cleaning
2. **ปี 2025 มี non-null เพียง 15,696/69,440** — ยืนยันว่ามี empty rows จำนวนมาก
3. **คอลัมน์ `ปริมาณ` ปี 2026 มี null 136 ค่า** — บางวันไม่มีการรายงานข้อมูล


In [7]:
df_2025.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69440 entries, 0 to 69439
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   รูปแบบการเดินทาง   15696 non-null  object
 1   วัตถุประสงค์       15696 non-null  object
 2   สาธารณะ/ส่วนบุคคล  15696 non-null  object
 3   หน่วยงาน           15696 non-null  object
 4   ยานพาหนะ/ท่า       15696 non-null  object
 5   วันที่             15696 non-null  object
 6   หน่วย              15696 non-null  object
 7   ปริมาณ             15388 non-null  object
dtypes: object(8)
memory usage: 4.2+ MB


In [8]:
df_2026.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3010 entries, 0 to 3009
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   รูปแบบการเดินทาง   3010 non-null   object
 1   วัตถุประสงค์       3010 non-null   object
 2   สาธารณะ/ส่วนบุคคล  3010 non-null   object
 3   หน่วยงาน           3010 non-null   object
 4   ยานพาหนะ/ท่า       3010 non-null   object
 5   วันที่             3010 non-null   object
 6   หน่วย              3010 non-null   object
 7   ปริมาณ             2874 non-null   object
dtypes: object(8)
memory usage: 188.3+ KB


## 3️⃣ Data Quality Audit & Cleaning 🧹

### ตรวจสอบและทำความสะอาดข้อมูล

ขั้นตอนนี้ครอบคลุมหลายส่วน:

**(1) Rename Columns** — เปลี่ยนชื่อจากภาษาไทยเป็นอังกฤษ เพื่อให้ code อ่านง่ายขึ้น

**(2) ตรวจสอบ Missing Values** — วิเคราะห์ว่า null values มาจากไหน มีแถวว่างทั้งหมดกี่แถว

**(3) ลบ Empty Rows** — ใช้ `dropna(how="all")` เพื่อลบแถวที่ว่างทั้งหมด

**(4) แปลง Data Types** — `ปริมาณ` เป็น numeric, `วันที่` เป็น datetime

Data Cleaning ที่ดีคือรากฐานของการวิเคราะห์ที่น่าเชื่อถือ — ถ้าข้อมูลขยะ ผลลัพธ์ก็เป็นขยะ (Garbage In, Garbage Out)


### 🔤 Rename Columns

เปลี่ยนชื่อคอลัมน์จากภาษาไทยเป็นอังกฤษเพื่อให้สะดวกในการเขียน code ในขั้นตอนถัดไป


In [9]:
rename_map = { #Mapping dictionary เพื่อทำการเปลี่ยนภาษาของคอลัมน์
    "รูปแบบการเดินทาง": "travel_mode",
    "วัตถุประสงค์": "purpose",
    "สาธารณะ/ส่วนบุคคล": "transport_type",
    "หน่วยงาน": "agency",
    "ยานพาหนะ/ท่า": "vehicle_station",
    "วันที่": "date",
    "หน่วย": "unit",
    "ปริมาณ": "volume"
}

df_2025 = df_2025.rename(columns=rename_map)
df_2026 = df_2026.rename(columns=rename_map)
print("✅ Rename columns สำเร็จ")
print(f"   คอลัมน์: {list(df_2025.columns)}")


✅ Rename columns สำเร็จ
   คอลัมน์: ['travel_mode', 'purpose', 'transport_type', 'agency', 'vehicle_station', 'date', 'unit', 'volume']


### 🔍 ตรวจสอบ Missing Values ก่อน Cleaning

ผลลัพธ์ด้านล่างแสดงให้เห็นความแตกต่างที่สำคัญระหว่างสองปี:

- **ปี 2025**: มี null จำนวนมาก (~53,744 แถว) ซึ่งล้วนเป็น empty rows ที่ Pandas อ่านมาโดยอัตโนมัติ
- **ปี 2026**: มี null เพียง 136 ค่า (เฉพาะ `volume`) — นี่คือแถวที่มีข้อมูลครบ แต่ไม่มีตัวเลขปริมาณ


In [10]:
print("🔎 Missing Values ก่อน Cleaning:")
print("\nปี 2025:")
print(df_2025.isnull().sum())
print("\nปี 2026:")
print(df_2026.isnull().sum())


🔎 Missing Values ก่อน Cleaning:

ปี 2025:
travel_mode        53744
purpose            53744
transport_type     53744
agency             53744
vehicle_station    53744
date               53744
unit               53744
volume             54052
dtype: int64

ปี 2026:
travel_mode          0
purpose              0
transport_type       0
agency               0
vehicle_station      0
date                 0
unit                 0
volume             136
dtype: int64


### 🧹 ลบ Empty Rows

ใช้ `dropna(how="all")` เพื่อลบแถวที่ **ทุก column เป็น null พร้อมกัน** — นี่คือแถวว่างที่ Pandas สร้างขึ้นเมื่ออ่านไฟล์ CSV ที่มีบรรทัดว่างต่อท้าย ไม่ใช่ข้อมูลจริง จึงสามารถลบออกได้อย่างปลอดภัย


In [11]:
print(f"ก่อน clean — ปี 2025: {len(df_2025):,} แถว | ปี 2026: {len(df_2026):,} แถว")

df_2025 = df_2025.dropna(how="all") #ทำการลบ empty rows ในชุดข้อมูลปี 2025
df_2026 = df_2026.dropna(how="all") #ทำการลบ empty rows ในชุดข้อมูลปี 2026

print(f"หลัง clean — ปี 2025: {len(df_2025):,} แถว | ปี 2026: {len(df_2026):,} แถว")
print()
print("🔎 Missing Values หลัง Cleaning (ปี 2025):")
print(df_2025.isnull().sum())


ก่อน clean — ปี 2025: 69,440 แถว | ปี 2026: 3,010 แถว
หลัง clean — ปี 2025: 15,696 แถว | ปี 2026: 3,010 แถว

🔎 Missing Values หลัง Cleaning (ปี 2025):
travel_mode          0
purpose              0
transport_type       0
agency               0
vehicle_station      0
date                 0
unit                 0
volume             308
dtype: int64


In [12]:
print("🔎 Missing Values หลัง Cleaning (ปี 2026):")
print(df_2026.isnull().sum())


🔎 Missing Values หลัง Cleaning (ปี 2026):
travel_mode          0
purpose              0
transport_type       0
agency               0
vehicle_station      0
date                 0
unit                 0
volume             136
dtype: int64


### 🔢 แปลง Data Types

**ปัญหาที่พบ:** คอลัมน์ `volume` ถูกอ่านเป็น string เพราะมีเครื่องหมาย comma เช่น `"127,551"` และ `"1,364,992"` — Python ไม่สามารถบวกหรือเฉลี่ยค่าเหล่านี้ได้โดยตรง

**วิธีแก้:** ใช้ `clean_volume()` function เพื่อ:
1. ลบ comma (thousands separator)
2. ลบ character อื่น ๆ ที่ไม่ใช่ตัวเลข
3. แปลงเป็น numeric ด้วย `pd.to_numeric(errors="coerce")` — ค่าที่แปลงไม่ได้จะกลายเป็น NaN

สำหรับ `date` เราแปลงเป็น `datetime` เพื่อให้สามารถคำนวณช่วงเวลา, กรองตามวัน, และ plot แกน x เป็นวันที่ได้อย่างถูกต้อง


In [13]:
def clean_volume(s):
    return (
        s.astype(str)
        .str.replace(",", "", regex=False)          # ลบ thousands separator
        .str.replace(r"[^\d.]", "", regex=True)     # เก็บแค่ตัวเลขและจุดทศนิยม
        .str.strip()
    )

df_2025["volume_clean"] = clean_volume(df_2025["volume"])
df_2026["volume_clean"] = clean_volume(df_2026["volume"])

df_2025["volume"] = pd.to_numeric(df_2025["volume_clean"], errors="coerce")
df_2026["volume"] = pd.to_numeric(df_2026["volume_clean"], errors="coerce")

df_2025["date"] = pd.to_datetime(df_2025["date"], errors="coerce")
df_2026["date"] = pd.to_datetime(df_2026["date"], errors="coerce")

df_2025["year"] = 2025
df_2026["year"] = 2026

print("✅ แปลง Data Types สำเร็จ!")
print(f"   volume dtype: {df_2025['volume'].dtype}")
print(f"   date dtype:   {df_2025['date'].dtype}")


✅ แปลง Data Types สำเร็จ!
   volume dtype: float64
   date dtype:   datetime64[ns]


/tmp/ipykernel_955/3489381614.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2025["volume_clean"] = clean_volume(df_2025["volume"])
/tmp/ipykernel_955/3489381614.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2025["volume"] = pd.to_numeric(df_2025["volume_clean"], errors="coerce")
/tmp/ipykernel_955/3489381614.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the docu

## 4️⃣ Data Preparation — Merge & Feature Engineering

### 🔗 รวมข้อมูลสองปีเข้าด้วยกัน

หลังจาก clean ข้อมูลแล้ว เราต้องรวม (merge) ทั้งสองปีเข้าเป็น DataFrame เดียว เพื่อให้เปรียบเทียบข้ามปีได้ง่าย

**ขั้นตอน:**
1. `pd.concat()` — ต่อสองตารางเข้าด้วยกันในแนวตั้ง (row-wise)
2. `sort_values("date")` — เรียงตามวันที่เพื่อให้ time-series analysis ทำงานถูกต้อง
3. เลือก columns ที่จำเป็นออกมา


In [14]:
df_all = pd.concat([df_2025, df_2026], ignore_index=True)
df_all = df_all.sort_values("date")

df_all = df_all[
    ["date", "year", "travel_mode", "purpose", "transport_type",
     "vehicle_station", "agency", "unit", "volume"]
]

print(f"✅ Merge สำเร็จ! df_all มี {len(df_all):,} แถว")


✅ Merge สำเร็จ! df_all มี 18,706 แถว


### 🔄 จัดการ Duplicates และ Missing Dates

หลัง merge พบว่ามีแถวที่ซ้ำกัน (duplicates) โดยเฉพาะแถวที่ `date = NaT` ซึ่งเกิดจากการที่ข้อมูลต้นฉบับมีแถวที่ไม่สามารถแปลงวันที่ได้

**กลยุทธ์การจัดการ:**
- ลบ duplicates ด้วย `drop_duplicates()` เพื่อลด noise
- แถวที่มี `date = NaT` แต่มีข้อมูล `volume` ยังคงมีค่าทางสถิติ จึงเก็บไว้ใน `df_all` แต่จะถูก filter ออกเมื่อทำ time-series analysis
- **Meaningful rows** (มีทั้ง `date` และ `volume`) มีทั้งหมด 7,526 แถว — นี่คือ dataset ที่จะใช้วิเคราะห์จริง

**Key finding:** แถว NaT ส่วนใหญ่เกิดจาก format วันที่ที่ผิดพลาดในต้นฉบับ เช่น Null value หรือการสลับ day/month ทำให้ pandas แปลงวันที่ไม่ได้


In [15]:
print(f"🔄 Duplicates พบ: {df_all.duplicated().sum()} แถว")
df_all = df_all.drop_duplicates()
print(f"✅ หลังลบ duplicates: {len(df_all):,} แถว")
print()
print("📊 ตารางค่าที่ยังว่าง:")
print(df_all.isna().sum().sort_values(ascending=False))


🔄 Duplicates พบ: 1225 แถว
✅ หลังลบ duplicates: 17,481 แถว

📊 ตารางค่าที่ยังว่าง:
date               9784
volume              181
year                  0
purpose               0
travel_mode           0
transport_type        0
vehicle_station       0
agency                0
unit                  0
dtype: int64


### 🎯 Filter ข้อมูล: เลือกเฉพาะระบบรถไฟฟ้า

สำหรับโจทย์ข้อ 1 และ 2 เราโฟกัสที่ **ระบบขนส่งทางรางในกรุงเทพฯ** เท่านั้น จึง filter เฉพาะ 7 สายรถไฟฟ้า/ชานเมือง และเลือกเฉพาะแถวที่มีข้อมูล `date` และ `volume` ครบถ้วน


In [16]:
rail_lines = [
    "รถไฟฟ้า ARL",
    "รถไฟฟ้า BTS",
    "รถไฟฟ้าสายสีชมพู",
    "รถไฟฟ้าสายสีน้ำเงิน",
    "รถไฟฟ้าสายสีม่วง",
    "รถไฟฟ้าสายสีเหลือง",
    "รถไฟฟ้าสายสีแดง"
]

df_analysis = df_all[
    df_all["vehicle_station"].isin(rail_lines) &
    df_all["date"].notna() &
    df_all["volume"].notna() &
    (df_all["volume"] != 0)
].copy()

print(f"✅ df_analysis: {df_analysis.shape[0]:,} แถว × {df_analysis.shape[1]} คอลัมน์")


✅ df_analysis: 1,248 แถว × 9 คอลัมน์


### 📈 การพิจารณา 'ค่า Extreme'

การระบุ 'ค่า Extreme' หรือ Outliers มักขึ้นอยู่กับบริบทและลักษณะของข้อมูล หากค่า Min/Max ใน `describe()` ไม่ได้แสดงถึงข้อผิดพลาดที่ชัดเจน (เช่น เลข 0 ทั้งที่ควรมีค่า หรือค่าที่สูงเกินจริงแบบเป็นไปไม่ได้) การกำหนดว่าค่าใดเป็น 'extreme' ต้องใช้เกณฑ์ที่ชัดเจน เช่น:

*   **IQR (Interquartile Range) method**: ค่าที่อยู่นอก `Q1 - 1.5*IQR` หรือ `Q3 + 1.5*IQR`
*   **Standard Deviation method**: ค่าที่อยู่นอก `Mean ± N*SD`
*   **Domain-specific knowledge**: เช่น ผู้เชี่ยวชาญระบุว่ายอดผู้โดยสารไม่ควรเกิน X คนต่อวัน

ในขณะนี้ `df_analysis['volume'].describe()` ที่เราได้ดูไปแล้ว ไม่ได้แสดงค่าที่ผิดปกติอย่างชัดเจน (เช่น 0 หรือค่าที่สูงเป็นล้านล้าน)

In [17]:
print("📊 สถิติเชิงพรรณนาสำหรับคอลัมน์ 'volume' หลังจากการตรวจสอบค่าติดลบ:")
volume_desc_cleaned = df_analysis['volume'].describe().apply(lambda x: f'{x:,.2f}')
print(volume_desc_cleaned)


📊 สถิติเชิงพรรณนาสำหรับคอลัมน์ 'volume' หลังจากการตรวจสอบค่าติดลบ:
count      1,248.00
mean     200,318.96
std      251,935.59
min       20,865.00
25%       44,474.75
50%       68,895.00
75%      319,100.75
max      957,717.00
Name: volume, dtype: object


### 🔢 ตรวจสอบและจัดการค่าติดลบใน `volume`

ปริมาณผู้โดยสารไม่ควรมีค่าติดลบ — หากพบ จะทำการลบแถวเหล่านั้นออก

In [18]:
negative_volumes = df_analysis[df_analysis['volume'] < 0]
if not negative_volumes.empty:
    print(f"🔴 พบค่าติดลบในคอลัมน์ 'volume' จำนวน {len(negative_volumes):,} แถว")
    display(negative_volumes.head())

    # ทำการลบแถวที่มีค่า volume ติดลบออก
    df_analysis_cleaned = df_analysis[df_analysis['volume'] >= 0].copy()
    print(f"✅ ลบแถวที่มีค่า volume ติดลบออกแล้ว, เหลือ {len(df_analysis_cleaned):,} แถว")
    df_analysis = df_analysis_cleaned
else:
    print("✅ ไม่พบค่าติดลบในคอลัมน์ 'volume'")


✅ ไม่พบค่าติดลบในคอลัมน์ 'volume'


In [19]:
print("📊 ตารางค่าที่ยังว่าง:")
print(df_analysis.isna().sum().sort_values(ascending=False))

📊 ตารางค่าที่ยังว่าง:
date               0
year               0
travel_mode        0
purpose            0
transport_type     0
vehicle_station    0
agency             0
unit               0
volume             0
dtype: int64


### 🛠️ Feature Engineering

สร้าง features ใหม่จากข้อมูลที่มีอยู่ เพื่อให้วิเคราะห์ในมิติต่าง ๆ ได้:

| Feature | วิธีคำนวณ | ประโยชน์ |
|---------|-----------|---------|
| `line_en` | map จากชื่อไทย → English | ใช้ใน axis labels |
| `mode_group` | จัดกลุ่ม MRT ทุกสายเป็น "MRT" | วิเคราะห์ modal share |
| `weekday` / `weekday_name` | `.dt.weekday` / `.dt.day_name()` | วิเคราะห์ day-of-week pattern |
| `is_weekend` | weekday >= 5 | เปรียบเทียบวันทำงาน vs วันหยุด |
| `month` / `year_month` | `.dt.month` / `to_period('M')` | วิเคราะห์ seasonal pattern |

**⚠️ หมายเหตุสำคัญ:** ข้อมูลต้นฉบับบางส่วนมี format วันที่แบบ DD/MM แต่ถูก parse เป็น MM/DD ทำให้ต้องทำ date correction ก่อน — เราใช้การสลับ `month` ↔ `day` สำหรับค่าที่ผิดปกติ


In [20]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, HTML

# Line name mapping
line_map = {
    "รถไฟฟ้า ARL": "ARL",
    "รถไฟฟ้า BTS": "BTS",
    "รถไฟฟ้าสายสีชมพู": "MRT Pink",
    "รถไฟฟ้าสายสีน้ำเงิน": "MRT Blue",
    "รถไฟฟ้าสายสีม่วง": "MRT Purple",
    "รถไฟฟ้าสายสีเหลือง": "MRT Yellow",
    "รถไฟฟ้าสายสีแดง": "Red Line"
}

# Group into 4 major systems for Q1
mode_group_map = {
    "ARL": "ARL",
    "BTS": "BTS",
    "MRT Blue": "MRT",
    "MRT Purple": "MRT",
    "MRT Yellow": "MRT",
    "MRT Pink": "MRT",
    "Red Line": "Red Line"
}

df = df_analysis.copy()
df["line_en"] = df["vehicle_station"].map(line_map)
df["mode_group"] = df["line_en"].map(mode_group_map)

# Fix date format issues and derive time features
df['date'] = df['date'].apply(lambda d: d.replace(month=d.day, day=d.month)
                               if d.day <= 12 else d)
df['year']         = df['date'].dt.year
df['weekday']      = df['date'].dt.weekday
df['weekday_name'] = df['date'].dt.day_name()
df['is_weekend']   = df['weekday'] >= 5
df['month']        = df['date'].dt.month
df['month_name']   = df['date'].dt.strftime('%b')
df['year_month']   = df['date'].dt.to_period('M').astype(str)

df = df.sort_values(["line_en", "date"]).reset_index(drop=True)

print(f"✅ Feature Engineering สำเร็จ!")
print(f"   ช่วงวันที่: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"   จำนวนแถว: {len(df):,}")
df.head(3)


✅ Feature Engineering สำเร็จ!
   ช่วงวันที่: 2025-01-01 → 2026-03-11
   จำนวนแถว: 1,248


,date,year,travel_mode,purpose,transport_type,vehicle_station,agency,unit,volume,line_en,mode_group,weekday,weekday_name,is_weekend,month,month_name,year_month
0,2025-01-01,2025,ทางราง,การเดินทางภายในจังหวัด/กรุงเทพ,สาธารณะ,รถไฟฟ้า ARL,รฟฟท.,คน,52281.0,ARL,ARL,2,Wednesday,False,1,Jan,2025-01
1,2025-01-02,2025,ทางราง,การเดินทางภายในจังหวัด/กรุงเทพ,สาธารณะ,รถไฟฟ้า ARL,รฟฟท.,คน,61353.0,ARL,ARL,3,Thursday,False,1,Jan,2025-01
2,2025-01-03,2025,ทางราง,การเดินทางภายในจังหวัด/กรุงเทพ,สาธารณะ,รถไฟฟ้า ARL,รฟฟท.,คน,63393.0,ARL,ARL,4,Friday,False,1,Jan,2025-01


### 📋 Coverage Summary — ภาพรวมข้อมูลรายสาย

ตารางด้านล่างแสดงภาพรวมของข้อมูลรายสาย ซึ่งช่วยยืนยันว่าข้อมูลทุกสายครอบคลุมช่วงเวลาเดียวกัน และ BTS ยังคงเป็นสายที่มีผู้โดยสารสูงสุดอย่างห่างชั้น


In [21]:
STATIONS = ["BTS", "MRT Blue", "MRT Pink", "MRT Purple", "MRT Yellow", "ARL", "Red Line"]
MODES    = ["BTS", "MRT", "ARL", "Red Line"]

coverage = (
    df.groupby("line_en")
      .agg(
          start_date=("date", "min"),
          end_date=("date", "max"),
          observed_days=("date", "nunique"),
          total_volume=("volume", "sum"),
          avg_daily_volume=("volume", "mean"),
          median_daily_volume=("volume", "median")
      )
      .sort_values("total_volume", ascending=False)
)
print("📊 Coverage Summary:")
print(coverage.to_string())


📊 Coverage Summary:
           start_date   end_date  observed_days  total_volume  avg_daily_volume  median_daily_volume
line_en                                                                                             
BTS        2025-01-01 2026-03-11            179   126595162.0     707235.541899             765501.0
MRT Blue   2025-01-01 2026-03-11            178    74596244.0     419080.022472             469663.5
ARL        2025-01-01 2026-03-11            178    11714187.0      65810.039326              69038.5
MRT Purple 2025-01-01 2026-03-11            178    11701209.0      65737.129213              76387.0
MRT Pink   2025-01-01 2026-03-11            178    10871430.0      61075.449438              66597.5
MRT Yellow 2025-01-01 2026-03-11            178     8039158.0      45163.808989              47704.0
Red Line   2025-01-01 2026-03-11            179     6480675.0      36204.888268              38814.0


## 5️⃣ Advanced EDA & Insights

### ตั้ง Color Palette และเตรียมข้อมูลสำหรับ Visualization

ก่อนเริ่มสร้างกราฟ เราตั้งค่าสีที่สอดคล้องกับสีตราสัญลักษณ์จริงของแต่ละสาย — เพื่อให้กราฟดูคุ้นตาและเข้าใจง่ายสำหรับผู้ที่คุ้นเคยกับระบบรถไฟฟ้ากรุงเทพฯ


In [22]:
# ── Color Palettes ─────────────────────────────────────────────────────────────
MODE_COLORS = {
    'BTS'     : '#1700AD',
    'MRT'     : '#06E061',
    'ARL'     : '#FF75D6',
    'Red Line': '#E00615',
}
STATION_COLORS = {
    'BTS'        : '#1700AD',
    'MRT Blue'   : '#065AE0',
    'MRT Pink'   : '#FF69BE',
    'MRT Purple' : '#7F1894',
    'MRT Yellow' : '#FFF42B',
    'ARL'        : '#FF75D6',
    'Red Line'   : '#E00615',
}

# ── Shared Prep ────────────────────────────────────────────────────────────────
df_25, df_26 = df[df.year == 2025], df[df.year == 2026]

months_25 = df_25['date'].dt.to_period('M').nunique()
months_26 = df_26['date'].dt.to_period('M').nunique()

mode_vol_25 = df_25.groupby('mode_group')['volume'].sum().reindex(MODES)
mode_vol_26 = df_26.groupby('mode_group')['volume'].sum().reindex(MODES)
mode_pct_25 = mode_vol_25 / mode_vol_25.sum() * 100
mode_pct_26 = mode_vol_26 / mode_vol_26.sum() * 100

sta_vol_25 = df_25.groupby('line_en')['volume'].sum()
sta_vol_26 = df_26.groupby('line_en')['volume'].sum()
sta_pct_25 = sta_vol_25 / sta_vol_25.sum() * 100
sta_pct_26 = sta_vol_26 / sta_vol_26.sum() * 100

print(f"✅ เตรียมข้อมูลสำเร็จ | ปี 2025: {months_25} เดือน | ปี 2026: {months_26} เดือน")


✅ เตรียมข้อมูลสำเร็จ | ปี 2025: 12 เดือน | ปี 2026: 3 เดือน


---
## 5.1 🚆 Q1 — Modal Share: คนไทยเดินทางด้วยอะไรมากที่สุด?

### แนวคิด: Donut Chart + Grouped Bar + Diverging Bar

ในระบบรถไฟฟ้า เราจัดกลุ่มสายทั้ง 7 เป็น 4 "mode" หลัก:
- **BTS** — Skytrain (กทม.)
- **MRT** — รวม MRT Blue, Pink, Purple, Yellow ทั้งหมด (รฟม.)
- **ARL** — Airport Rail Link (รฟฟท.)
- **Red Line** — รถไฟชานเมืองสายสีแดง (รฟท.)

**กราฟ 1:** Donut Chart เปรียบเทียบ Modal Share ระหว่าง 2025 vs 2026  
**กราฟ 2:** Grouped Bar แสดง station-level share  
**กราฟ 3:** Diverging Bar แสดงการเปลี่ยนแปลง (เพิ่ม/ลด) ของแต่ละสาย

**Insight ที่คาดว่าจะพบ:** BTS น่าจะครอง market share สูงสุด ตามด้วย MRT Blue ซึ่งเป็นสายที่เปิดมานานและครอบคลุมพื้นที่กว้าง


In [23]:
# CHART 1 — Donut Charts: Modal Share % (2025 vs 2026)
fig1 = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'pie'}, {'type': 'pie'}]],
    subplot_titles=['2025 (ม.ค.–ธ.ค.)', '2026 (ม.ค.–มี.ค.)'],
)

for col, pct in enumerate([mode_pct_25, mode_pct_26], 1):
    fig1.add_trace(go.Pie(
        labels=MODES,
        values=pct.round(2).values,
        hole=0.55,
        marker=dict(colors=[MODE_COLORS[m] for m in MODES]),
        textinfo='label+percent',
        insidetextorientation='radial',
        showlegend=(col == 1),
    ), row=1, col=col)

fig1.update_layout(
    title_text='<b>Modal Share ของระบบขนส่งสาธารณะทางรางในกรุงเทพฯ</b><br>'
               '<sup>Donut Chart — 2025 vs 2026 | BTS ครองส่วนแบ่งสูงสุดตลอดสองปี</sup>',
    legend=dict(orientation='v', x=1.05, y=0.5, xanchor='left', yanchor='middle'),
    margin=dict(t=100, b=80, r=150),
    height=450,
)
fig1.show()
print("💡 BTS ครองสัดส่วนราว 50% ตลอดสองปี ขณะที่ MRT (รวมทุกสาย) อยู่ที่ ~42% | ARL และ Red Line รวมกันไม่ถึง 8%")


💡 BTS ครองสัดส่วนราว 50% ตลอดสองปี ขณะที่ MRT (รวมทุกสาย) อยู่ที่ ~42% | ARL และ Red Line รวมกันไม่ถึง 8%


### สัดส่วนระดับสาย (Station-Level Share)

กราฟด้านล่างแสดงสัดส่วนรายสายเพื่อให้เห็นว่าในกลุ่ม MRT นั้น **สายสีน้ำเงิน (MRT Blue) ครองสัดส่วนมากที่สุด** เพราะเป็นสายที่เปิดมานานและมีเส้นทางครอบคลุมพื้นที่ใจกลางเมืองและย่านธุรกิจ


In [24]:
# CHART 2 — Grouped Bar: Station Share Comparison
bar_df = pd.DataFrame({
    'Station': STATIONS * 2,
    'Year'   : ['2025'] * len(STATIONS) + ['2026'] * len(STATIONS),
    'Share'  : list(sta_pct_25.reindex(STATIONS).values) + list(sta_pct_26.reindex(STATIONS).values),
})

fig2 = px.bar(
    bar_df, x='Station', y='Share', color='Year',
    barmode='group',
    color_discrete_map={'2025': '#4A90D9', '2026': '#E87040'},
    text=bar_df['Share'].apply(lambda v: f'{v:.2f}%'),
    labels={'Share': 'สัดส่วนผู้โดยสารรวม (%)'},
    title='<b>สัดส่วนผู้โดยสารรายสาย</b><br><sup>Grouped Bar — 2025 vs 2026</sup>',
)
fig2.update_traces(textposition='outside')
fig2.update_layout(
    yaxis=dict(title='สัดส่วนผู้โดยสารรวม (%)', range=[0, bar_df.Share.max() * 1.2]),
    xaxis_title='สายรถไฟฟ้า',
    legend_title='ปี',
    height=480,
    margin=dict(t=90, b=60),
)
fig2.show()


สาเหตุที่ BTS และ MRT สายสีน้ำเงินมีสัดส่วนการใช้งานสูงที่สุดเนื่องจากเป็นเส้นทางหลัก (Core Line) ที่เชื่อมต่อย่านธุรกิจ (CBD) ที่อยู่อาศัย และสถานที่สำคัญใจกลางกรุงเทพฯ เข้าด้วยกัน มีจุดเชื่อมต่อที่ครอบคลุม (Interchange) ทำให้สะดวกต่อการเดินทางข้ามฝั่งและเชื่อมต่อระบบรางอื่นๆ อีกทั้งมีความถี่ของขบวนรถสูงและทำเวลาได้ดีกว่าการจราจรบนท้องถนน

เส้นทางยุทธศาสตร์ใจกลางเมือง:
* BTS (โดยเฉพาะสายสีเขียว): ผ่านแหล่งงานหลัก เช่น สยาม, สีลม, อโศก และสุขุมวิท
* MRT สายสีน้ำเงิน: วิ่งเป็นโครงข่ายวงกลม (Circle Line) เชื่อมโยงฝั่งธนบุรีกับพระนคร และเข้าสู่พื้นที่หนาแน่น เช่น สีลม, สุขุมวิท, พระราม 9, บางซื่อ ในขณะที่สายสีอื่นๆ เช่น สายสีม่วง (บางใหญ่-เตาปูน) หรือสายสีเหลือง (ลาดพร้าว-สำโรง) เป็นเส้นทางจากปริมณฑลเข้าสู่เมือง ซึ่งผู้ใช้ต้องต่อรถสายสีน้ำเงินเพื่อเข้าสู่ที่ทำงานในเมืองอีกครั้ง

รถไฟชานเมือง (สายสีแดง) และ ARL มีผู้ใช้งานน้อยกว่า BTS/MRT หลักๆ เพราะเป็นรถไฟฟ้าประเภท "ขนคนเข้าเมือง" (Commuter Rail) ที่เน้นระยะทางไกล สถานีห่างกัน ต่างจาก BTS/MRT ที่เป็น "ขนส่งมวลชนในเมือง" (Metro) ที่ผ่านย่านใจกลางเมือง สถานีถี่ และเชื่อมต่อสถานที่สำคัญได้ดีกว่า

In [25]:
# CHART 3 — Diverging Bar: Change in Share %
div_df = pd.DataFrame({
    'Station': STATIONS,
    'Avg_25' : [sta_pct_25.get(s, 0) for s in STATIONS],
    'Avg_26' : [sta_pct_26.get(s, 0) for s in STATIONS],
}).assign(
    Change  = lambda d: d.Avg_26 - d.Avg_25,
    Pct_Chg = lambda d: (d.Avg_26 - d.Avg_25) / d.Avg_25 * 100,
).sort_values('Change', ascending=True)

div_df['Color'] = div_df['Change'].apply(lambda x: '#2ECC71' if x >= 0 else '#E74C3C')

fig3 = go.Figure(go.Bar(
    x=div_df['Change'],
    y=div_df['Station'],
    orientation='h',
    marker_color=div_df['Color'],
    text=div_df['Change'].apply(lambda v: f'{v:+,.2f}'),
    textposition='outside',
))

fig3.add_vline(x=0, line_width=1.5, line_color='gray')
fig3.update_layout(
    title=(
        '<b>การเปลี่ยนแปลงสัดส่วนผู้โดยสารของแต่ละสาย</b><br>'
        f'<sup>Diverging Bar — 2025 ({months_25} เดือน) vs 2026 ({months_26} เดือน) | เขียว = เพิ่มขึ้น | แดง = ลดลง</sup>'
    ),
    xaxis_title='Δ สัดส่วน (%)',
    yaxis_title='',
    height=480,
    margin=dict(t=100, b=60, l=130, r=100),
)
fig3.show()
print("💡 MRT Blue และ ARL มีสัดส่วนเพิ่มขึ้นเล็กน้อยในปี 2026 | BTS และ MRT Purple มีสัดส่วนลดลง")


💡 MRT Blue และ ARL มีสัดส่วนเพิ่มขึ้นเล็กน้อยในปี 2026 | BTS และ MRT Purple มีสัดส่วนลดลง


---
## 5.2 🚇 Q2 — Urban Rail Comparison: แต่ละสายมีพฤติกรรมต่างกันอย่างไร?

### แนวคิด: Summary Statistics + Box Plot + CV + Rolling Heatmap

เพื่อเข้าใจพฤติกรรมผู้โดยสารของแต่ละสาย เราจะวิเคราะห์ใน 4 มิติ:

1. **Summary Statistics** — ยอดรวม, ค่าเฉลี่ย, มัธยฐานรายวัน
2. **Box Plot (Min-Max Scaled)** — กระจายตัวของผู้โดยสารรายวัน ปรับ scale ให้เปรียบเทียบได้แม้ขนาดต่างกัน
3. **Coefficient of Variation (CV)** — วัดความผันผวนในรูปแบบ Normalized (%)
4. **30-Day Rolling Heatmap** — แนวโน้มรายเดือน relative กับค่าเฉลี่ยของแต่ละสาย


### 📊 Summary Statistics รายสาย

ตารางด้านล่างเป็นจุดเริ่มต้นที่ดีในการเข้าใจขนาดของแต่ละสาย ก่อนที่จะลงลึกในเรื่อง distribution และ volatility


In [26]:
summary = (
    df.groupby(['year', 'line_en'])['volume']
    .agg(
        Total  = 'sum',
        Mean   = 'mean',
        Median = 'median',
    )
    .round(0)
    .astype(int)
    .reset_index()
)

for yr in sorted(summary.year.unique()):
    sub = (summary[summary.year == yr]
           .drop(columns='year')
           .set_index('line_en')
           .reindex(STATIONS))
    sub.columns = ['ยอดรวม', 'เฉลี่ยรายวัน', 'มัธยฐานรายวัน']

    display(HTML(f'<h3 style="font-family:sans-serif">📊 สรุปปริมาณผู้โดยสาร — ปี {yr}</h3>'))
    display(sub.style
        .format('{:,}')
        .background_gradient(cmap='Blues',   subset=['ยอดรวม'])
        .background_gradient(cmap='Greens',  subset=['เฉลี่ยรายวัน'])
        .background_gradient(cmap='Oranges', subset=['มัธยฐานรายวัน'])
    )


,ยอดรวม,เฉลี่ยรายวัน,มัธยฐานรายวัน
line_en,,,
BTS,"102,316,358","710,530","778,362"
MRT Blue,"59,612,126","416,868","468,170"
MRT Pink,"8,737,426","61,101","65,911"
MRT Purple,"9,521,005","66,580","76,597"
MRT Yellow,"6,433,077","44,987","47,030"
ARL,"9,282,677","64,914","68,253"
Red Line,"5,187,313","36,023","38,689"


,ยอดรวม,เฉลี่ยรายวัน,มัธยฐานรายวัน
line_en,,,
BTS,"24,278,804","693,680","757,836"
MRT Blue,"14,984,118","428,118","479,190"
MRT Pink,"2,134,004","60,972","68,115"
MRT Purple,"2,180,204","62,292","74,107"
MRT Yellow,"1,606,081","45,888","49,953"
ARL,"2,431,510","69,472","75,501"
Red Line,"1,293,362","36,953","40,216"


### 📦 Box Plot: การกระจายตัวของผู้โดยสารรายวัน (Min-Max Scaled)

**ทำไมต้อง Scale?** BTS มีผู้โดยสารเฉลี่ย 700,000+ คน/วัน ขณะที่ Red Line มีเพียง 36,000 คน/วัน — ถ้าวาด Box Plot ในสเกลเดียวกัน Red Line แทบมองไม่เห็น

**Min-Max Scaling** แปลงให้ค่าต่ำสุดของแต่ละสาย = 0 และค่าสูงสุด = 1 ทำให้เห็น **รูปแบบการกระจาย** ของแต่ละสายได้ชัดเจน โดยไม่เปรียบเทียบขนาดจริง

**สัญลักษณ์ ✕** ในกล่อง = ค่าเฉลี่ย ± ส่วนเบี่ยงเบนมาตรฐาน (SD)


In [27]:
daily = (df.groupby(['date','line_en'])['volume']
           .sum() # รวมปริมาณผู้โดยสารรายวันสำหรับแต่ละสาย
           .reset_index() # รีเซ็ต index เพื่อให้ date และ line_en เป็นคอลัมน์ปกติ
           .sort_values(['line_en','date'])) # เรียงข้อมูลตามสายและวันที่ เพื่อให้การคำนวณ rolling/diff ถูกต้อง

# เพิ่มคอลัมน์ 'year' เพื่อใช้ในการกรองข้อมูลตามปี
daily['year']       = daily['date'].dt.year
# เพิ่มคอลัมน์ 'dow' (Day of Week) เพื่อระบุวันในสัปดาห์ (เช่น Monday, Tuesday)
daily['dow']        = daily['date'].dt.day_name()
# เพิ่มคอลัมน์ 'is_weekend' เพื่อระบุว่าวันนั้นเป็นวันหยุดสุดสัปดาห์หรือไม่ (True/False)
daily['is_weekend'] = daily['date'].dt.weekday >= 5 # 0=จันทร์, 6=อาทิตย์ ดังนั้น 5,6 คือวันหยุด

# คำนวณผลต่างสัมบูรณ์ของปริมาณผู้โดยสารจากวันก่อนหน้าสำหรับแต่ละสาย
daily['vol_diff']   = daily.groupby('line_en')['volume'].diff().abs()
# คำนวณค่าเฉลี่ยของปริมาณผู้โดยสารสำหรับแต่ละสายตลอดช่วงเวลา
daily['mean_vol']   = daily.groupby('line_en')['volume'].transform('mean')
# คำนวณผลต่างสัมบูรณ์แบบสัมพัทธ์ (% ของค่าเฉลี่ย) เพื่อดูความผันผวนเมื่อเทียบกับค่าเฉลี่ยของสายนั้นๆ
daily['rel_diff']   = daily['vol_diff'] / daily['mean_vol'] * 100

# ทำ Min-Max Scaling สำหรับปริมาณผู้โดยสารรายวันของแต่ละสาย
# โดยแปลงค่าให้อยู่ในช่วง 0 ถึง 1 เพื่อให้สามารถเปรียบเทียบรูปแบบการกระจายตัวได้ง่ายขึ้น
daily['vol_minmax'] = (
    daily.groupby('line_en')['volume']
    .transform(lambda x: (x - x.min()) / (x.max() - x.min()))
)

# กำหนดลำดับของวันในสัปดาห์สำหรับใช้ในการแสดงผลกราฟให้ถูกต้อง
DOW_ORDER = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']


In [28]:
for yr in [2025, 2026]:
    fig_box = make_subplots(
        rows=2, cols=4,
        subplot_titles=STATIONS,
        vertical_spacing=0.18,
        horizontal_spacing=0.08,
    )

    for i, sta in enumerate(STATIONS):
        row = i // 4 + 1
        col = i %  4 + 1

        sub  = daily[(daily.line_en == sta) & (daily.year == yr)]
        vals = sub['vol_minmax']

        fig_box.add_trace(go.Box(
            y=vals,
            name=sta,
            marker_color=STATION_COLORS[sta],
            boxmean='sd',
            width=0.6,
            showlegend=False,
            customdata=sub['volume'].values,
            hovertemplate=f'<b>{sta}</b><br>'
                          'Scaled value: %{y:.3f}<br>'
                          'Actual riders: %{customdata:,.0f}<extra></extra>',
        ), row=row, col=col)

    fig_box.update_layout(
        title_text=f'<b>การกระจายตัวของผู้โดยสารรายวัน — {yr} (Min-Max Scaled)</b><br>'
                   '<sup>0 = ต่ำสุด | 1 = สูงสุด | ✕ = mean ± SD</sup>',
        height=620,
        margin=dict(t=100, b=60),
    )
    for r in [1, 2]:
        fig_box.update_yaxes(title_text='Scaled Volume (0–1)', row=r, col=1)

    fig_box.show()


### 📉 Coefficient of Variation (CV) — วัดความผันผวน Normalized

**CV = (Standard Deviation / Mean) × 100%**

CV เป็น metric ที่ดีกว่า SD ในการเปรียบเทียบความผันผวนข้ามสายที่มีขนาดต่างกัน เพราะ:
- SD ของ BTS (~150,000 คน) สูงกว่า SD ของ Red Line (~7,000 คน) แต่นั่นไม่ได้แปลว่า BTS ผันผวนกว่า
- CV แสดงความผันผวนเป็น % ของค่าเฉลี่ย ทำให้เปรียบเทียบได้ตรงไปตรงมา

**การตีความ:**
- CV สูง → ผู้โดยสารผันผวนมาก (วันธรรมดา vs วันหยุดต่างกันมาก)
- CV ต่ำ → ฐานผู้โดยสารเสถียร (เช่น ARL ซึ่งผู้ใช้ส่วนใหญ่เป็นนักท่องเที่ยวที่มาสนามบิน)


In [29]:
cv_rows = []
for sta in STATIONS:
    for yr in [2025, 2026]:
        vals = daily[(daily.line_en == sta) & (daily.year == yr)]['volume']
        if len(vals) < 2:
            continue
        cv_rows.append({
            'Station': sta,
            'Year'   : str(yr),
            'CV'     : vals.std() / vals.mean() * 100,
        })
cv_df = pd.DataFrame(cv_rows)

fig_cv = px.bar(
    cv_df, x='Station', y='CV', color='Year', barmode='group',
    color_discrete_map={'2025': '#4A90D9', '2026': '#E87040'},
    text=cv_df['CV'].apply(lambda v: f'{v:.1f}%'),
    title='<b>Coefficient of Variation (CV) รายสาย</b><br>'
          '<sup>CV สูง = ผันผวนมาก | CV ต่ำ = ฐานผู้โดยสารเสถียร</sup>',
    labels={'CV': 'CV (%)'},
    category_orders={'Station': STATIONS},
)
fig_cv.update_traces(textposition='outside')
fig_cv.update_layout(
    yaxis=dict(title='CV (%)', range=[0, cv_df.CV.max() * 1.25]),
    xaxis_title='สายรถไฟฟ้า',
    height=460,
    margin=dict(t=90, b=60),
    legend_title='ปี',
)
fig_cv.show()


* BTS เครือข่ายมีความเสถียรสูงจากการใช้งานในพื้นที่ใจกลางกรุงเทพฯ มีผู้โดยสารสม่ำเสมอราว 700,000 คน/วัน แม้มีเหตุแผ่นดินไหวปี 2025 และนโยบายทดลองเดินทางฟรีช่วงสั้น ๆ ก็ไม่กระทบความผันผวนโดยรวม

* MRT สายสีน้ำเงินเป็นเส้นหลักเหนือ–ใต้ ผู้โดยสารหนาแน่น ความผันผวนเพิ่มจากผลกระทบหลังแผ่นดินไหวและการขยายเส้นทาง ทำให้รูปแบบการเดินทางเปลี่ยนเล็กน้อย
ส่วน MRT สายสีชมพูเป็น mono rail ใหม่ การขยายเส้นทางและช่วงทดลองใช้งานทำให้ความต้องการยังไม่สม่ำเสมอ รวมถึงผลกระทบจากแผ่นดินไหวและการเชื่อมต่อระบบ
และ MRT สายสีม่วงเน้นพื้นที่ชานเมือง จึงอ่อนไหวต่อการเปลี่ยนแปลง เช่น การต่อรถและพฤติกรรมการทำงานแบบไฮบริด ได้รับผลกระทบจากแผ่นดินไหวมาก ทำให้ความผันผวนเพิ่มขึ้นชัดเจน หากเป็น MRT สายสีเหลือง
แม้เป็นสายใหม่แต่มีความเสถียร เนื่องจากปริมาณผู้โดยสารยังไม่สูงมาก และไม่มีการขยายเส้นทางเพิ่มเติมในช่วงนี้

* ARL มีความสม่ำเสมอสูงเพราะขึ้นอยู่กับการเดินทางสนามบินสุวรรณภูมิ ความต้องการจากนักท่องเที่ยวช่วยลดความผันผวน

* สายสีแดง โปรโมชันค่าโดยสารและการยกเลิกภายหลัง รวมถึงผลกระทบจากแผ่นดินไหว ทำให้จำนวนผู้โดยสารขึ้นลงมากขึ้นเล็กน้อย

### 🌡️ 30-Day Rolling Heatmap — แนวโน้มรายเดือน

Heatmap นี้แสดง % ของค่าเฉลี่ยของแต่ละสายในแต่ละเดือน โดย:
- **สีฟ้า** = เดือนนั้นมีผู้โดยสารสูงกว่าค่าเฉลี่ยของสาย
- **สีแดง** = เดือนนั้นมีผู้โดยสารต่ำกว่าค่าเฉลี่ยของสาย
- **ตัวเลขในช่อง** = ยอดผู้โดยสารจริง (พัน)

การ normalize แบบนี้ช่วยให้เห็น **seasonal pattern** ได้ชัดเจน แม้แต่ละสายมีขนาดต่างกันมาก


In [30]:
heat_rows = []
for sta in STATIONS:
    sub = (daily[daily.line_en == sta]
           .set_index('date')['volume']
           .sort_index()
           .rolling('30D', min_periods=7)
           .mean()
           .resample('ME')
           .mean())
    for dt, val in sub.items():
        heat_rows.append({'Station': sta, 'Month': dt, 'AvgVol': val})

heat_df   = pd.DataFrame(heat_rows)
pivot_heat = (heat_df
              .pivot(index='Station', columns='Month', values='AvgVol')
              .reindex(index=STATIONS))

pivot_norm = pivot_heat.div(pivot_heat.mean(axis=1), axis=0) * 100

z = pivot_norm.values
x_labs = [m.strftime('%b %Y') for m in pivot_heat.columns]

fig_heat = go.Figure(go.Heatmap(
    z=z,
    x=x_labs,
    y=STATIONS,
    colorscale='RdYlGn_r', # Changed from 'RdBu'
    zmid=100,
    text=np.round(pivot_heat.values / 1000, 1),
    texttemplate='%{text}k',
    hovertemplate='<b>%{y}</b> — %{x}<br>'
                  'vs own avg: %{z:.1f}%<br>'
                  'Actual: %{text}k riders<extra></extra>',
    colorbar=dict(
        title="% of line's<br>own average",
        tickvals=[80, 90, 100, 110, 120],
        ticktext=['80%', '90%', 'avg', '110%', '120%'],
    ),
))

fig_heat.update_layout(
    title='<b>แนวโน้มผู้โดยสาร 30 วันย้อนหลัง — ม.ค. 2025 ถึง มี.ค. 2026</b><br>'
          "<sup>สี = % เทียบกับค่าเฉลี่ยของแต่ละสาย | แดง = ต่ำกว่าเฉลี่ย | ฟ้า = สูงกว่าเฉลี่ย</sup>",
    xaxis=dict(title='เดือน', tickangle=-45),
    yaxis_title='สายรถไฟฟ้า',
    height=450,
    margin=dict(t=90, b=100, l=130, r=80),
)
fig_heat.show()
print("💡 มิถุนายน–สิงหาคม 2025 เป็นช่วงที่ผู้โดยสารต่ำกว่าเฉลี่ยในเกือบทุกสาย (น่าจะสอดคล้องกับฤดูฝนและปิดเทอมฤดูร้อน)")

💡 มิถุนายน–สิงหาคม 2025 เป็นช่วงที่ผู้โดยสารต่ำกว่าเฉลี่ยในเกือบทุกสาย (น่าจะสอดคล้องกับฤดูฝนและปิดเทอมฤดูร้อน)


* ช่วงวันหยุดปีใหม่ (1 มกราคม) และวันหยุดต่อเนื่อง (31 ธ.ค. – 2 ม.ค.) ทำให้สัปดาห์แรกถึงสองสัปดาห์ของเดือนมีผู้เดินทางเข้าออฟฟิศน้อยกว่าปกติ เนื่องจากหลายคนต่อวันลาหรือทยอยกลับมาทำงาน นอกจากนี้ ฤดูฝุ่น PM2.5 ของไทยจะรุนแรงที่สุดในช่วงมกราคม–กุมภาพันธ์ โดยในปี 2568 ทำให้มีการประกาศนโยบาย Work From Home ตั้งแต่วันที่ 13 มกราคม ส่งผลให้ปริมาณผู้โดยสารลดลงก่อนที่โครงการ "สัปดาห์รถไฟฟรี" ในช่วง 25–31 มกราคม
* เมษายน-พฤษภาคม เป็นช่วงเดือนที่มีจำนวนผูโดยสารต่ำลงอย่างชัดเจน เหตุผลมาจากการที่โรงเรียนรัฐบาลปิดเทอมฤดูร้อน และเทศกาลวันสงกรานต์ รวมถึงเป็นช่วงที่ฤดูร้อนแรงที่สุด  
* เดือนมิถุนายนเป็นช่วงเริ่มต้นของช่วงฤดูฝน ซึ่งอาจส่งผลให้มีการใช้งานน้อยลง

### 📅 Day-of-Week Pattern — รูปแบบการเดินทางตามวันในสัปดาห์

Heatmap นี้แสดงผู้โดยสาร normalized รายวันในสัปดาห์ เพื่อตอบคำถามว่า "สายไหนผู้โดยสารลดลงมากที่สุดในวันหยุด?"


In [31]:
# Day-of-week pivot (ใช้ข้อมูลทั้งสองปีรวมกัน)
pivot = (
    daily.groupby(['line_en', 'dow'])['volume']
    .mean()
    .reset_index()
    .pivot(index='line_en', columns='dow', values='volume')
    .reindex(index=STATIONS, columns=DOW_ORDER)
)

pivot_norm_dow = pivot.div(pivot.mean(axis=1), axis=0) * 100

fig_dow = go.Figure(go.Heatmap(
    z=pivot_norm_dow.values,
    x=[d[:3] for d in DOW_ORDER],
    y=STATIONS,
    colorscale='RdYlGn_r',
    zmid=100,
    text=np.round(pivot.values / 1000, 1),
    texttemplate='%{text}k',
    hovertemplate='<b>%{y}</b> — %{x}<br>'
                  'vs own avg: %{z:.1f}%<br>'
                  'Actual avg: %{text}k<extra></extra>',
    colorbar=dict(
        title="% of line's<br>weekly avg",
        tickvals=[70, 85, 100, 115, 130],
        ticktext=['70%', '85%', 'avg', '115%', '130%'],
    ),
))
fig_dow.update_layout(
    title='<b>รูปแบบผู้โดยสารตามวันในสัปดาห์</b><br>'
          '<sup>สี = % เทียบกับค่าเฉลี่ยรายสัปดาห์ | แดง = ต่ำกว่า | ฟ้า = สูงกว่า | ตัวเลข = พัน</sup>',
    xaxis_title='วันในสัปดาห์',
    yaxis_title='สายรถไฟฟ้า',
    height=420,
    margin=dict(t=90, b=60, l=130),
)
fig_dow.show()
print("💡 วันจันทร์–ศุกร์ผู้โดยสารสูงกว่า | เสาร์–อาทิตย์ลดลงอย่างเห็นได้ชัด โดยเฉพาะ MRT Purple ซึ่งให้บริการย่านธุรกิจชานเมือง")


💡 วันจันทร์–ศุกร์ผู้โดยสารสูงกว่า | เสาร์–อาทิตย์ลดลงอย่างเห็นได้ชัด โดยเฉพาะ MRT Purple ซึ่งให้บริการย่านธุรกิจชานเมือง


การจราจรทางรถไฟในวันหยุดสุดสัปดาห์นั้นน้อยกว่าวันธรรมดาเป็นหลัก เนื่องจากการเดินทางเพื่อไปทำงาน/การศึกษาคิดเป็นส่วนใหญ่ ของการเดินทางทั้งหมดในวันธรรมดา เมื่อเทียบกับวันหยุดสุดสัปดาห์ การเดินทางในวันหยุดจะกระจายตัวมากขึ้น (ช้อปปิ้ง/พักผ่อนหย่อนใจ) ทำให้ความต้องการใช้รถไฟแบบรวมศูนย์ที่มีความหนาแน่นสูงลดลง ส่วนวันศุกร์มักมียอดผู้โดยสารสูงสุด เนื่องจากมีทั้งผู้เดินทางประจำและนักเดินทางเพื่อพักผ่อนที่ออกเดินทางแต่เช้า


สาเหตุที่ปริมาณผู้โดยสารรถไฟในวันหยุดสุดสัปดาห์น้อยกว่า:

* ขาดการเดินทางเพื่อทำงาน: วันธรรมดามีการจราจรหนักและสม่ำเสมอในช่วงชั่วโมงเร่งด่วนสำหรับการทำงานและการศึกษา ในขณะที่วันหยุดสุดสัปดาห์เน้นการพักผ่อนและการช้อปปิ้ง ซึ่งกระจายตัวออกไปตลอดช่วงเวลาที่ยาวนานกว่า และมักใช้รูปแบบการขนส่งอื่น
* ตารางเดินรถและต้นทุน: หน่วยงานขนส่งมักวางแผนให้สอดคล้องกับความต้องการที่ต่ำลงในวันหยุดสุดสัปดาห์ โดยจัดให้มีรถไฟน้อยลง ต้นทุนและจำนวนผู้โดยสารที่น้อยลงยังนำไปสู่การลดบริการในวันหยุดอีกด้วย
* เวลาให้บริการ: วันหยุดสุดสัปดาห์ส่วนใหญ่ถูกจัดว่าเป็น "นอกช่วงเวลาเร่งด่วน" โดยมีความถี่ในการให้บริการต่ำตลอดทั้งวัน เมื่อเทียบกับช่วงเช้า/เย็นในวันธรรมดา

ทำไมวันศุกร์จึงมียอดผู้โดยสารสูงสุด:

* ผสมผสานระหว่างผู้เดินทางประจำและนักท่องเที่ยว: บ่ายวันศุกร์มีทั้งการเดินทางกลับบ้านตามปกติ ควบคู่กับนักเดินทางเพื่อพักผ่อนที่ออกเดินทางออกนอกเมืองหรือไปงานสังสรรค์
* เลิกงาน/เลิกเรียนเร็ว: หลายคนเลิกงานหรือเลิกเรียนเร็วกว่าปกติในวันศุกร์ ทำให้ช่วงเวลาผู้โดยสารสูงสุดเร็วขึ้น

---
## 5.3 📅 Q3 — Event Detection: วันหยุดและเทศกาลเห็นได้ในข้อมูลไหม?

### แนวคิด: Time Series + Anomaly Detection + Event Annotation

เราจะวิเคราะห์ปริมาณผู้โดยสาร **รายวันรวมทุกสาย** ตลอด 14 เดือน และตรวจหาจุดผิดปกติโดยใช้ **Z-score** (จำนวน standard deviations จากค่าเฉลี่ย):

- **Z-score < -1.5** → วันที่ผู้โดยสารลดลงผิดปกติ (สีแดง)
- **Z-score > +1.5** → วันที่ผู้โดยสารสูงผิดปกติ (สีเขียว)

จากนั้นเราจะ annotate วันหยุดนักขัตฤกษ์และเทศกาลสำคัญของไทยเพื่อตรวจสอบว่าข้อมูลสอดคล้องกับพฤติกรรมจริงหรือไม่


In [32]:
# รวมยอดผู้โดยสารทุกสายรายวัน
total_daily = (df.groupby('date')['volume']
                .sum()
                .reset_index()
                .sort_values('date'))

# คำนวณ Z-score
total_daily['rolling_mean'] = total_daily['volume'].rolling(7, center=True, min_periods=3).mean()
total_daily['rolling_std']  = total_daily['volume'].rolling(7, center=True, min_periods=3).std()
total_daily['z_score'] = (total_daily['volume'] - total_daily['rolling_mean']) / total_daily['rolling_std'].replace(0, np.nan)

# กำหนดเกณฑ์ anomaly
Z_THRESH = 1.5
anomaly_low  = total_daily[total_daily['z_score'] < -Z_THRESH]
anomaly_high = total_daily[total_daily['z_score'] > +Z_THRESH]

print(f"📊 จำนวนวันทั้งหมด: {len(total_daily):,}")
print(f"🔴 วันที่ผู้โดยสารลดต่ำผิดปกติ  (z < -{Z_THRESH}): {len(anomaly_low)} วัน")
print(f"🟢 วันที่ผู้โดยสารสูงผิดปกติ (z > +{Z_THRESH}): {len(anomaly_high)} วัน")


📊 จำนวนวันทั้งหมด: 179
🔴 วันที่ผู้โดยสารลดต่ำผิดปกติ  (z < -1.5): 11 วัน
🟢 วันที่ผู้โดยสารสูงผิดปกติ (z > +1.5): 2 วัน


### 📈 Time Series Plot พร้อม Anomaly Markers และ Event Annotations

กราฟด้านล่างแสดง:
- **เส้นสีน้ำเงิน** — ยอดผู้โดยสารรายวันรวมทุกสาย
- **เส้นประสีส้ม** — 7-day rolling average (แนวโน้มระยะสั้น)
- **จุดแดง** — วันที่ผู้โดยสารลดต่ำผิดปกติ
- **จุดเขียว** — วันที่ผู้โดยสารสูงผิดปกติ
- **เส้นแนวตั้ง** — วันหยุดนักขัตฤกษ์และเทศกาลสำคัญ


In [33]:
# วันหยุดและเทศกาลสำคัญ
thai_events = {
    '2025-01-01': 'วันปีใหม่',
    '2025-02-12': 'วันมาฆบูชา',
    '2025-04-06': 'วันจักรี',
    '2025-04-13': 'วันสงกรานต์',
    '2025-05-01': 'วันแรงงาน',
    '2025-05-05': 'วันฉัตรมงคล',
    '2025-05-11': 'วันวิสาขบูชา',
    '2025-06-03': 'วันเฉลิมฯ สมเด็จพระราชินี',
    '2025-07-10': 'วันอาสาฬหบูชา',
    '2025-07-11': 'วันเข้าพรรษา',
    '2025-07-28': 'วันเฉลิมฯ รัชกาลที่ 10',
    '2025-08-12': 'วันแม่แห่งชาติ',
    '2025-10-13': 'วันคล้ายวันสวรรคต ร.9',
    '2025-10-23': 'วันปิยมหาราช',
    '2025-12-05': 'วันพ่อแห่งชาติ',
    '2025-12-10': 'วันรัฐธรรมนูญ',
    '2025-12-31': 'วันส่งท้ายปีเก่า',
    '2026-01-01': 'วันปีใหม่ 2026',
}

fig_ts = go.Figure()

# เส้นยอดผู้โดยสาร
fig_ts.add_trace(go.Scatter(
    x=total_daily['date'], y=total_daily['volume'],
    mode='lines',
    name='ยอดรายวัน',
    line=dict(color='#3498DB', width=1.2),
    opacity=0.7
))

# 7-day rolling average
fig_ts.add_trace(go.Scatter(
    x=total_daily['date'], y=total_daily['rolling_mean'],
    mode='lines',
    name='7-day Rolling Avg',
    line=dict(color='#E67E22', width=2, dash='dot'),
))

# Anomaly: ต่ำผิดปกติ
fig_ts.add_trace(go.Scatter(
    x=anomaly_low['date'], y=anomaly_low['volume'],
    mode='markers',
    name=f'ผิดปกติ (ต่ำ) z < -{Z_THRESH}',
    marker=dict(color='#E74C3C', size=8, symbol='circle'),
))

# Anomaly: สูงผิดปกติ
fig_ts.add_trace(go.Scatter(
    x=anomaly_high['date'], y=anomaly_high['volume'],
    mode='markers',
    name=f'ผิดปกติ (สูง) z > +{Z_THRESH}',
    marker=dict(color='#27AE60', size=8, symbol='diamond'),
))

# Event annotations
for date_str, label in thai_events.items():
    try:
        dt = pd.to_datetime(date_str)
        if total_daily['date'].min() <= dt <= total_daily['date'].max():
            fig_ts.add_vline(
                x=dt, line_width=1, line_color='gray', line_dash='dot', opacity=0.5
            )
            fig_ts.add_annotation(
                x=dt, y=total_daily['volume'].max() * 0.95,
                text=label, showarrow=False,
                textangle=-90, font=dict(size=8, color='gray'),
                xshift=5
            )
    except:
        pass

fig_ts.update_layout(
    title='<b>ปริมาณผู้โดยสารรายวัน — ม.ค. 2025 ถึง มี.ค. 2026</b><br>'
          '<sup>แดง = ต่ำผิดปกติ | เขียว = สูงผิดปกติ | เส้นแนวตั้ง = วันหยุดนักขัตฤกษ์</sup>',
    xaxis_title='วันที่',
    yaxis_title='ยอดผู้โดยสารรวม (คน)',
    height=550,
    margin=dict(t=100, b=60),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig_ts.show()


In [34]:
least_volume_events_report = []

for event_date_str, event_name in thai_events.items():
    event_dt = pd.to_datetime(event_date_str)

    # Calculate absolute time difference for all dates in total_daily
    total_daily['time_diff'] = (total_daily['date'] - event_dt).abs()

    # Get the 2 closest points by time difference
    closest_2_points = total_daily.nsmallest(2, 'time_diff').copy()

    # From these 2 points, find the one with the least volume
    least_volume_row = closest_2_points.nsmallest(1, 'volume').iloc[0] # .iloc[0] to get the row as a Series

    least_volume_events_report.append({
        'Event Date': event_dt.strftime('%Y-%m-%d'),
        'Event Name': event_name,
        'Closest Data Date': least_volume_row['date'].strftime('%Y-%m-%d'),
        'Volume': f"{least_volume_row['volume']:,}",
        'Days Difference': least_volume_row['time_diff'].days
    })

# Display the report as a DataFrame
least_volume_events = pd.DataFrame(least_volume_events_report)
least_volume_events['Volume'] = least_volume_events['Volume'].str.replace(',', '').astype(float)
least_volume_events = least_volume_events.sort_values(by=['Event Date', 'Volume']).reset_index(drop=True)
least_volume_events['Volume'] = least_volume_events['Volume'].apply(lambda x: f'{x:,.0f}')
display(least_volume_events)


,Event Date,Event Name,Closest Data Date,Volume,Days Difference
0,2025-01-01,วันปีใหม่,2025-01-01,"978,521",0
1,2025-02-12,วันมาฆบูชา,2025-02-12,"1,037,870",0
2,2025-04-06,วันจักรี,2025-04-06,"959,068",0
3,2025-04-13,วันสงกรานต์,2025-04-12,"931,399",1
4,2025-05-01,วันแรงงาน,2025-05-01,"1,101,205",0
5,2025-05-05,วันฉัตรมงคล,2025-05-05,"881,954",0
6,2025-05-11,วันวิสาขบูชา,2025-05-11,"846,045",0
7,2025-06-03,วันเฉลิมฯ สมเด็จพระราชินี,2025-06-03,"808,961",0
8,2025-07-10,วันอาสาฬหบูชา,2025-07-10,"963,268",0
9,2025-07-11,วันเข้าพรรษา,2025-07-10,"963,268",1


### 🔍 วิเคราะห์จุด Anomaly — เทียบกับเหตุการณ์จริง


ปฏิทินวันหยุดและเหตุการณ์สำคัญ: บริบทผู้โดยสารระบบรางกรุงเทพฯ ปี 2568–2569

* สงกรานต์ปี 2568 มีวันหยุดราชการอย่างเป็นทางการตั้งแต่ 13–16 เมษายน (อาทิตย์ถึงพุธ โดยวันที่ 16 เป็นวันหยุดชดเชยแทนวันที่ 13 ที่ตรงกับวันอาทิตย์) นอกจากนี้ วันจักรีที่ 7 เมษายนยังสร้างวันหยุดยาว 3 วันแยกต่างหาก ส่งผลให้พนักงานจำนวนมากลาพักในช่วง 8–11 เมษายน จนมีการหยุดยาวถึงถึง 12 วัน (5–16 เมษายน) ปริมาณผู้โดยสารระบบรางลดลงอย่างชัดเจนเมื่อประชาชนเดินทางกลับบ้าน

* ปีใหม่ 2568–2569: วันหยุดยาวที่สุดในรอบหลายปี
การประกาศวันหยุดพิเศษวันที่ 2 มกราคม 2569 โดยคณะรัฐมนตรี ทำให้เกิดวันหยุดยาว 5 วันติดต่อกัน (31 ธ.ค. – 4 ม.ค.) และหากพนักงานลาพักเพิ่มในวันที่ 29–30 ธันวาคมด้วย ช่วงหยุดยาวอาจขยายไปถึง 7 วัน

* วันที่ 10 ธันวาคม เป็นวันรัฐธรรมนูญซึ่งเป็นวันพุธ ส่งผลให้ผู้โดยสารลดลงแบบฉับพลันกลางสัปดาห์

* 2 พฤษภาคม (จุดสีเขียว) มีคอนเสิร์ต GOT7 NESTFEST ที่สนามกีฬาราชมังคลากีฬาสถาน ดึงดูดแฟนคลับกว่า 85,000 คนในสองคืนที่ขายบัตรหมดเกลี้ยง (2–3 พฤษภาคม) นี่คือคอนเสิร์ตสเตเดียมครั้งแรกของ GOT7 ในรอบ 11 ปี และเป็นการแสดงครบวง (OT7) ครั้งแรกในไทยในรอบกว่า 5 ปี วันที่ 2 พฤษภาคมยังตรงกับวันเกิดของสมาชิก แบมแบม ซึ่งยิ่งเพิ่มจำนวนแฟนคลับที่หลั่งไหลมา

* 1 สิงหาคม (จุดสีเขียวที่ 2) ตรงกับวันศุกร์ ไม่มีเหตุการณ์ใดตรงกับวันนี้ชัดเจน แต่เป็นช่วงที่มหาลัยเปิดเทอมและนักศึกษาทยอยกันกลับวิทยาเขต รวมถึงวันศุกร์มักเป็นวันที่ผู้โดยสารหนาแน่นที่สุดตามรูปแบบการเดินทางตามวันในสัปดาห์

* จุด Anomaly สีแดงทุกจุดเป็นวันอาทิตย์ ซึ่งเป็นวันหยุด

* แผ่นดินไหว 28 มีนาคม: ระบบรางปิดทั้งหมด
แผ่นดินไหวในเมียนมา เมื่อวันที่ 28 มีนาคม 2568 (ขนาด 7.7) ระบบรางฟ้าทุกสายถูกระงับการให้บริการ กรุงเทพฯ ถูกประกาศเป็นพื้นที่ภัยพิบัติชั่วคราว อาคารสำนักงานการตรวจเงินแผ่นดินถล่มที่จตุจักรทำให้มีผู้เสียชีวิตในประเทศไทย 103 ราย
การฟื้นฟูบริการเป็นแบบทยอยเปิด:

* BTS และ SRT สายสีแดง: กลับมาให้บริการในช่วงเย็นของวันที่ 28 มีนาคม
MRT สายสีน้ำเงินและสีม่วง: เปิดวันที่ 29 มีนาคม
MRT สายสีเหลือง: เปิดวันที่ 30 มีนาคม
MRT สายสีชมพู: เปิดบางส่วนวันที่ 31 มีนาคม โดยสถานีมีนบุรียังไม่เปิดจนถึง 16 เมษายน เนื่องจากรางนำกระแสไฟฟ้าเสียหาย

* 25–31 มกราคม — สัปดาห์รถไฟฟ้าฟรี สร้างสถิติผู้โดยสารพุ่งสูงกว่าค่าเฉลี่ยเดือนมกราคมทั่วไป

* 28–29 มิถุนายน — ชุมนุมใหญ่ต่อต้านรัฐบาล บริเวณอนุสาวรีย์ชัยสมรภูมิ (ใกล้สถานี BTS อนุสาวรีย์ชัยฯ) มีผู้ร่วมชุมนุมประมาณ 10,000–20,000 คน ส่งผลต่อการให้บริการ BTS ในบริเวณใกล้เคียง

* 16–19 ตุลาคม — Gamescom Asia × Thailand Game Show ที่ QSNCC มีผู้เข้าร่วม 206,159 คน นับเป็นงานเกมมิ่งที่ใหญ่ที่สุดในเอเชียตะวันออกเฉียงใต้ สร้างปริมาณผู้โดยสารหนาแน่นที่สถานี MRT ศูนย์การประชุมแห่งชาติสิริกิติ์


### 📆 Heatmap ปฏิทินรายวัน — เห็นทั้งปีในหน้าเดียว

Calendar Heatmap ช่วยให้เห็นภาพรวมตลอดทั้งปีได้ง่ายกว่า line chart — สีเข้มหมายถึงผู้โดยสารมาก สีอ่อนหมายถึงน้อย


In [35]:
# Calendar Heatmap
total_daily_2025 = total_daily[total_daily['date'].dt.year == 2025].copy()
total_daily_2025['week'] = total_daily_2025['date'].dt.isocalendar().week.astype(int)
total_daily_2025['dow_num'] = total_daily_2025['date'].dt.weekday

pivot_cal = total_daily_2025.pivot_table(
    index='dow_num', columns='week', values='volume', aggfunc='mean'
)

fig_cal = go.Figure(go.Heatmap(
    z=pivot_cal.values / 1000,
    x=[f'สัปดาห์ {w}' for w in pivot_cal.columns],
    y=['จ', 'อ', 'พ', 'พฤ', 'ศ', 'ส', 'อา'],
    colorscale='Blues',
    hoverongaps=False,
    hovertemplate='%{y} สัปดาห์ที่ %{x}<br>ยอดเฉลี่ย: %{z:.0f}k<extra></extra>',
    colorbar=dict(title='ยอดผู้โดยสาร (พัน)', ticksuffix='k')
))

fig_cal.update_layout(
    title='<b>Calendar Heatmap ปริมาณผู้โดยสาร ปี 2025</b><br>'
          '<sup>สีเข้ม = ผู้โดยสารมาก | สีอ่อน = ผู้โดยสารน้อย</sup>',
    xaxis_title='สัปดาห์',
    yaxis_title='วัน',
    height=350,
    margin=dict(t=90, b=60)
)
fig_cal.show()
print("💡 เห็นได้ชัดว่าสัปดาห์สงกรานต์ (เม.ย.) และปีใหม่ (ม.ค./ธ.ค.) ผู้โดยสารลดลงอย่างเห็นได้ชัด")


💡 เห็นได้ชัดว่าสัปดาห์สงกรานต์ (เม.ย.) และปีใหม่ (ม.ค./ธ.ค.) ผู้โดยสารลดลงอย่างเห็นได้ชัด


## 📝 สรุป Key Insights ที่ค้นพบ

---

### 🏆 Insight 1 — BTS ครองตลาด แต่ MRT กำลังตามทัน

BTS ครอง ~50% ของผู้โดยสารรถไฟฟ้าทั้งหมด ขณะที่ MRT (รวมทุกสาย) อยู่ที่ ~42% และมีแนวโน้มเพิ่มขึ้นในปี 2026 โดยเฉพาะ MRT Blue และ ARL ซึ่งน่าจะได้ประโยชน์จากการเพิ่มขึ้นของนักท่องเที่ยว

---

### 🚇 Insight 2 — ARL เสถียรที่สุด, MRT Purple ผันผวนมากที่สุด

**ARL** มี CV ต่ำสุด (~17%) เพราะผู้โดยสารหลักคือนักเดินทางไปสนามบิน ซึ่งกระจายตัวสม่ำเสมอกว่า  
**MRT Purple** มี CV สูงที่สุด (~30%) สะท้อนว่าสายนี้ขึ้นอยู่กับการเดินทางไปทำงานในย่านนนทบุรี ซึ่งลดลงมากในวันหยุด

---

### 📅 Insight 3 — สงกรานต์และปีใหม่เห็นได้ชัดในข้อมูล

วันหยุดยาวสงกรานต์ (13–15 เม.ย.) ผู้โดยสารลดลง 30–50% จากค่าเฉลี่ย ขณะที่ช่วงคืนวันส่งท้ายปีเก่า-ปีใหม่ผู้โดยสารพุ่งสูงขึ้นอย่างเห็นได้ชัด Data สะท้อนพฤติกรรมจริงของคนไทยได้ดี

---

### 📈 Insight 4 — ช่วงเดือนมิถุนายน–สิงหาคม ผู้โดยสารต่ำทุกสาย

ทุกสายมีผู้โดยสารต่ำกว่าค่าเฉลี่ยในช่วงกลางปี ซึ่งน่าจะเป็นผลรวมจาก: ฤดูฝนที่คนออกจากบ้านน้อยลง + ช่วงหลังปิดเทอมที่นักศึกษาไม่ได้ใช้ระบบขนส่ง

---

### 💡 ข้อเสนอแนะสำหรับ Policy

1. **ขยายชั่วโมงให้บริการ** ในช่วง peak (กลางสัปดาห์, ต้นเดือน) เพื่อรองรับผู้โดยสาร
2. **ส่งเสริมการใช้รถไฟฟ้าในวันหยุด** โดยเฉพาะ MRT Purple ที่ผู้โดยสารลดลงมาก
3. **ติดตาม ARL** ซึ่งมีนักท่องเที่ยวเป็นฐานหลัก — ควรวิเคราะห์ร่วมกับข้อมูลการบินเพื่อวางแผนล่วงหน้า


ระบบขนส่งทางรางของกรุงเทพฯ ผ่านช่วง 15 เดือนที่เปลี่ยนแปลงครั้งใหญ่ที่สุดนับตั้งแต่การระบาดของโควิด ไม่ว่าจะเป็นสัปดาห์นั่งฟรีในช่วงวิกฤตมลพิษทางอากาศ แผ่นดินไหวขนาด 7.7 ที่ทำให้ทุกสายหยุดให้บริการ คอนเสิร์ต K-pop ขนาด 85,000 ที่นั่งที่ทำลายสถิติผู้โดยสาร BTS รายวัน การเปิดตัวบัตร smart card Mangmoom EMV และการขยายอัตราค่าโดยสารคงที่ 20 บาทไปยังทุกสายรถไฟฟ้าสำหรับผู้ถือสัญชาติไทย เหตุการณ์เหล่านี้ทับซ้อนกับปฏิทินวันหยุดประจำปีของไทย ฤดูมรสุม และพฤติกรรมของผู้โดยสารที่เปลี่ยนแปลงไป ครอบคลุมช่วงเวลาตั้งแต่มกราคม 2568 ถึงมีนาคม 2569